# Modulo 8 · Lezione 4 — TensorFlow e Keras: Architettura di Base

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vediamo oggi?

| # | Argomento |
|---|------|
| 1 | Cosa sono TensorFlow e Keras? |
| 2 | Tensori: il dato fondamentale |
| 3 | API Sequential: costruire modelli |
| 4 | Compilare e addestrare |
| 5 | Valutare, predire e salvare |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
gpus = tf.config.list_physical_devices('GPU')
print(f"GPU disponibili: {len(gpus)} {'(addestramento accelerato)' if gpus else '(modalità CPU)'}")

---
## 1. Cosa sono TensorFlow e Keras?

```
TENSORFLOW (Google Brain, 2015):
  Framework di calcolo numerico per ML/DL.
  Calcola AUTOMATICAMENTE i gradienti (differenziazione automatica).
  Esegue operazioni su CPU, GPU o TPU senza cambiare il codice.
  Libreria di basso livello → controllo totale, ma verbosa.

KERAS (François Chollet, 2015 → dentro TF dalla 2.0):
  API di alto livello sopra TensorFlow.
  Progettata per essere semplice e leggibile.
  Permette di costruire reti in poche righe.
  "Deep learning for humans"

Relazione:
  TensorFlow  →  il motore
  Keras       →  il volante e i pedali
  import tensorflow as tf
  from tensorflow import keras

Alternative:
  PyTorch (Meta): più diffuso nella ricerca
  JAX (Google): per la ricerca all'avanguardia
  sklearn: per il ML classico (non DL)

Quando usare TF/Keras:
  → produzione, API web, mobile, hardware specializzato
  → progetti con dati di immagine, audio, testo
  → quando vuoi codice stabile e ben documentato
```

---
## 2. Tensori: il dato fondamentale

In [ ]:
# I tensori sono come array NumPy ma con supporto GPU e gradienti automatici

# Creare tensori
escalar   = tf.constant(3.14)                          # 0D: un numero
vector    = tf.constant([1.0, 2.0, 3.0])               # 1D: vettore
matriz    = tf.constant([[1.0, 2.0], [3.0, 4.0]])       # 2D: matrice
tensor_3d = tf.constant(np.random.randn(2, 3, 4))       # 3D: batch di matrici

print("Tensori:")
for nombre, t in [('scalare',escalar),('vettore',vector),('matrice',matriz),('3D',tensor_3d)]:
    print(f"  {nombre:10}: shape={t.shape}, dtype={t.dtype.name}")

# Operazioni (uguale a NumPy)
a = tf.constant([1.0, 2.0, 3.0])
b = tf.constant([4.0, 5.0, 6.0])
print(f"\na + b     = {(a+b).numpy()}")
print(f"a * b     = {(a*b).numpy()}")
print(f"a @ b T   = {tf.tensordot(a, b, axes=1).numpy():.1f}")
print(f"tf.reduce_mean = {tf.reduce_mean(a).numpy():.1f}")

# Convertire in NumPy
t = tf.constant([[1.0, 2.0], [3.0, 4.0]])
arr = t.numpy()
print(f"\nTensore → NumPy: {type(arr).__name__}, shape={arr.shape}")

In [ ]:
# Differenziazione automatica: TF calcola i gradienti senza backprop manuale
w = tf.Variable(0.5)   # Variable: parametro che TF può derivare
x = tf.constant(2.0)
y = tf.constant(3.0)   # target

with tf.GradientTape() as tape:
    y_hat = w * x           # predizione
    loss  = (y_hat - y)**2  # MSE su 1 campione

grad = tape.gradient(loss, w)  # ∂loss/∂w automaticamente

print("Differenziazione automatica:")
print(f"  w={w.numpy()}, x={x.numpy()}, y={y.numpy()}")
print(f"  y_hat = w·x = {(w*x).numpy():.2f}")
print(f"  loss  = (y_hat - y)² = {loss.numpy():.4f}")
print(f"  ∂loss/∂w (automatico)  = {grad.numpy():.4f}")
print(f"  ∂loss/∂w (manuale) = 2(wx-y)x = {2*(w.numpy()*x.numpy()-y.numpy())*x.numpy():.4f}")
print("  → coincidono! TF ha calcolato la derivata senza che noi scrivessimo nulla.")

---
## 3. API Sequential: costruire modelli

In [ ]:
# API Sequential: il modo più semplice per creare modelli in Keras

# Modello per classificazione binaria (come quello della Lezione 3, ma in 3 righe)
modelo = keras.Sequential([
    layers.Input(shape=(2,)),            # strato di ingresso: definisce la forma
    layers.Dense(8, activation='relu'),  # strato nascosto: 8 neuroni, ReLU
    layers.Dense(4, activation='relu'),  # strato nascosto: 4 neuroni, ReLU
    layers.Dense(1, activation='sigmoid'), # uscita: 1 neurone, Sigmoid (binaria)
])

# Riepilogo del modello
modelo.summary()

In [ ]:
# Gli strati più importanti di Keras
print("Tipi di strati in Keras:")
print("""
layers.Dense(units, activation)     → strato completamente connesso
layers.Input(shape)                 → definisce la forma dell'ingresso
layers.Dropout(rate)                → regolarizzazione (Lezione 6)
layers.BatchNormalization()         → normalizzazione per batch (Lezione 6)
layers.Conv2D(filters, kernel_size) → convoluzione 2D (Lezione 7)
layers.MaxPooling2D(pool_size)      → pooling (Lezione 7)
layers.Flatten()                    → appiattisce il tensore a 1D
layers.LSTM(units)                  → strato ricorrente (Lezione 8)
layers.Embedding(vocab, dim)        → embedding di testo (Lezione 8)

Attivazioni: 'relu', 'sigmoid', 'tanh', 'softmax', 'linear'
""")

# Diversi modelli di esempio
print("Modello multiclasse (3 classi):")
modelo_multi = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(3,  activation='softmax'),  # softmax per il multiclasse
])
modelo_multi.summary()

print("\nModello di regressione:")
modelo_reg = keras.Sequential([
    layers.Input(shape=(5,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),  # senza attivazione = lineare → per la regressione
])
modelo_reg.summary()

---
## 4. Compilare e addestrare

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Dati
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

# 1. Creare il modello
tf.random.set_seed(42)
modelo = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])

# 2. Compilare: definire optimizer, loss e metriche
modelo.compile(
    optimizer='adam',                          # oppure: keras.optimizers.Adam(lr=0.001)
    loss='binary_crossentropy',                # BCE per classificazione binaria
    metrics=['accuracy']                       # metriche da monitorare
)

# 3. Addestrare
historial = modelo.fit(
    X_tr_n, y_tr,
    epochs=50,             # numero di epoche
    batch_size=32,         # dimensione del mini-batch
    validation_split=0.2,  # 20% del train come validazione
    verbose=1              # 1: mostra il progresso, 0: silenzioso
)

In [ ]:
# Visualizzare la cronologia dell'addestramento
def plot_historial(historial, titulo=''):
    hist = historial.history
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    axes[0].plot(hist['loss'],     'steelblue', lw=2, label='Train loss')
    axes[0].plot(hist['val_loss'], 'coral',     lw=2, label='Val loss')
    axes[0].set_xlabel('Epoca'); axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Perdita{" — " + titulo if titulo else ""}')
    axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(hist['accuracy'],     'steelblue', lw=2, label='Train acc')
    axes[1].plot(hist['val_accuracy'], 'coral',     lw=2, label='Val acc')
    axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title(f'Accuracy{" — " + titulo if titulo else ""}')
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()

plot_historial(historial, 'Make Moons')

---
## 5. Valutare, predire e salvare

In [ ]:
# Valutare sul test
loss_te, acc_te = modelo.evaluate(X_te_n, y_te, verbose=0)
print(f"Test  → Loss: {loss_te:.4f}, Accuracy: {acc_te:.4f}")

# Predire probabilità
probas = modelo.predict(X_te_n[:5], verbose=0)
clases = (probas >= 0.5).astype(int).ravel()

print(f"\nPrimi 5 campioni del test:")
print(f"  Prob classe 1: {probas.ravel()[:5].round(4)}")
print(f"  Classe predetta: {clases}")
print(f"  Classe reale:     {y_te[:5]}")

# Frontiera di decisione
x1_range = np.linspace(X_te_n[:,0].min()-0.5, X_te_n[:,0].max()+0.5, 200)
x2_range = np.linspace(X_te_n[:,1].min()-0.5, X_te_n[:,1].max()+0.5, 200)
XX, YY = np.meshgrid(x1_range, x2_range)
Z = modelo.predict(np.c_[XX.ravel(), YY.ravel()], verbose=0).reshape(XX.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(XX, YY, Z, alpha=0.3, cmap='RdBu')
ax.contour(XX, YY, Z, levels=[0.5], colors='black', linewidths=2)
c_map = ['coral' if yi==0 else 'steelblue' for yi in y_te]
ax.scatter(X_te_n[:,0], X_te_n[:,1], c=c_map, s=25, edgecolors='black', lw=0.5, zorder=5)
ax.set_title(f'Frontiera di decisione — Keras (acc={acc_te:.2%})')
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
plt.tight_layout(); plt.show()

In [ ]:
# Salvare e caricare modelli
# Formato consigliato: SavedModel (cartella) o .keras
modelo.save('modelo_moons.keras')
print("Modello salvato come 'modelo_moons.keras'")

# Caricare
modelo_cargado = keras.models.load_model('modelo_moons.keras')
loss_c, acc_c = modelo_cargado.evaluate(X_te_n, y_te, verbose=0)
print(f"Modello caricato → Accuracy: {acc_c:.4f}  (uguale a prima)")

# Salvare solo i pesi
modelo.save_weights('pesos_moons.weights.h5')
print("Pesi salvati come 'pesos_moons.weights.h5'")

In [ ]:
# Callback: azioni automatiche durante l'addestramento
tf.random.set_seed(42)

callbacks = [
    # EarlyStopping: fermarsi quando val_loss non migliora
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,         # aspettare 10 epoche senza miglioramento prima di fermarsi
        restore_best_weights=True  # ripristinare i pesi migliori
    ),
    # ModelCheckpoint: salvare il modello migliore
    keras.callbacks.ModelCheckpoint(
        filepath='mejor_modelo.keras',
        monitor='val_loss',
        save_best_only=True
    ),
    # ReduceLROnPlateau: ridurre il lr se si stagna
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,       # moltiplicare il lr × 0.5
        patience=5,
        min_lr=1e-6
    ),
]

modelo2 = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])
modelo2.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

historial2 = modelo2.fit(
    X_tr_n, y_tr,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=0
)

n_epocas = len(historial2.history['loss'])
loss_te2, acc_te2 = modelo2.evaluate(X_te_n, y_te, verbose=0)
print(f"Addestrato per {n_epocas} epoche (su 200 possibili) — EarlyStopping attivato")
print(f"Test accuracy: {acc_te2:.4f}")
plot_historial(historial2, 'con EarlyStopping')

---
## Pratica

### Esercizio 1 — Primo modello Keras su MNIST

**Consegna:** MNIST è l'"Hello World" del Deep Learning: 70000 immagini di cifre scritte a mano.

```python
(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
# X: (N, 28, 28), y: (N,) con classi 0-9
```

1. Normalizza: `X = X / 255.0` e appiattisci: `X = X.reshape(-1, 784)`
2. Costruisci un modello Sequential: 784 → 128 → 64 → 10 (softmax)
3. Compila: `optimizer='adam'`, `loss='sparse_categorical_crossentropy'`
4. Addestra per 10 epoche, batch=64
5. Che accuracy ottiene sul test?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0
print(f"Train: {X_tr.shape}, Test: {X_te.shape}")

tf.random.set_seed(42)
modelo = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(10,  activation='softmax'),  # 10 classi → softmax
])

modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # y sono interi (non one-hot)
    metrics=['accuracy']
)
modelo.summary()

hist = modelo.fit(X_tr, y_tr, epochs=10, batch_size=64,
                  validation_split=0.1, verbose=1)

loss, acc = modelo.evaluate(X_te, y_te, verbose=0)
print(f"\nTest accuracy: {acc:.4f} ({acc:.2%})")

# Vedere alcune predizioni
y_pred = modelo.predict(X_te[:9], verbose=0).argmax(axis=1)
fig, axes = plt.subplots(3, 3, figsize=(7, 7))
for ax, i in zip(axes.ravel(), range(9)):
    ax.imshow(X_te[i].reshape(28,28), cmap='gray')
    ok = y_pred[i] == y_te[i]
    ax.set_title(f'Pred:{y_pred[i]}  Reale:{y_te[i]}',
                 color='green' if ok else 'red')
    ax.axis('off')
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 2 — Effetto dell'architettura

**Consegna:** Con il dataset MNIST, confronta 3 architetture diverse:

- Modello A: 784 → 32 → 10 (piccolo)
- Modello B: 784 → 256 → 128 → 10 (medio)
- Modello C: 784 → 512 → 256 → 128 → 64 → 10 (grande)

1. Addestra ciascuno per 5 epoche
2. Confronta: accuracy sul test, tempo di addestramento, numero di parametri
3. Quale ha il miglior rapporto qualità/costo?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
import time
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

(X_tr, y_tr), (X_te, y_te) = keras.datasets.mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0

arquitecturas = [
    ('A (piccolo)', [32]),
    ('B (medio)',   [256, 128]),
    ('C (grande)',  [512, 256, 128, 64]),
]

print(f"{'Modello':15} {'Params':>10} {'Acc test':>10} {'Tempo (s)':>12}")
print("-" * 52)

for nombre, capas_ocultas in arquitecturas:
    tf.random.set_seed(42)
    capas = [layers.Input(shape=(784,))]
    for n in capas_ocultas:
        capas.append(layers.Dense(n, activation='relu'))
    capas.append(layers.Dense(10, activation='softmax'))

    m = keras.Sequential(capas)
    m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

    t0 = time.time()
    m.fit(X_tr, y_tr, epochs=5, batch_size=64, verbose=0)
    t1 = time.time()

    _, acc = m.evaluate(X_te, y_te, verbose=0)
    params = m.count_params()
    print(f"{nombre:15} {params:>10,} {acc:>10.4f} {t1-t0:>12.1f}")
```
</details>

---
### Esercizio 3 — Callback in azione

**Consegna:** Addestra un modello con EarlyStopping e ReduceLROnPlateau:

1. Dataset: `keras.datasets.fashion_mnist` (abbigliamento invece di cifre)
2. Architettura: 784 → 256 → 128 → 10 (softmax)
3. EarlyStopping(patience=8, restore_best_weights=True)
4. ReduceLROnPlateau(factor=0.5, patience=4)
5. Plotta la curva di loss e segnala dove ha ridotto il lr

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

(X_tr, y_tr), (X_te, y_te) = keras.datasets.fashion_mnist.load_data()
X_tr = X_tr.reshape(-1, 784) / 255.0
X_te = X_te.reshape(-1, 784) / 255.0

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(784,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(128, activation='relu'),
    layers.Dense(10,  activation='softmax'),
])
m.compile('adam', 'sparse_categorical_crossentropy', ['accuracy'])

lr_hist = []
class LRLogger(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        lr_hist.append(float(self.model.optimizer.learning_rate))

cbs = [
    keras.callbacks.EarlyStopping('val_loss', patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau('val_loss', factor=0.5, patience=4, min_lr=1e-6),
    LRLogger(),
]

hist = m.fit(X_tr, y_tr, epochs=100, batch_size=64, validation_split=0.1,
             callbacks=cbs, verbose=0)

_, acc = m.evaluate(X_te, y_te, verbose=0)
print(f"Epoche addestrate: {len(hist.history['loss'])}")
print(f"Test accuracy: {acc:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(hist.history['loss'],     label='Train loss')
axes[0].plot(hist.history['val_loss'], label='Val loss')
# Segnalare i cambi di lr
for i in range(1, len(lr_hist)):
    if lr_hist[i] < lr_hist[i-1]:
        axes[0].axvline(i, color='red', ls='--', alpha=0.6)
axes[0].set_title('Loss (linee rosse = riduzione lr)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(lr_hist, 'orange', lw=2)
axes[1].set_title('Learning rate per epoca')
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('lr')
axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
```
</details>

---
### Esercizio 4 — Regressione con Keras

**Consegna:** Usa California Housing con Keras:

```python
from sklearn.datasets import fetch_california_housing
data = fetch_california_housing()
X, y = data.data, data.target
```

1. Normalizza X con StandardScaler
2. Modello: 8 → 64 → 32 → 1 (senza attivazione finale)
3. Compila con `loss='mse'`, `metrics=['mae']`
4. Addestra con EarlyStopping
5. Riporta il MAE sul test (in 100k $)

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

data = fetch_california_housing()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

tf.random.set_seed(42)
m = keras.Sequential([
    layers.Input(shape=(8,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),   # senza attivazione → regressione
])
m.compile(optimizer=keras.optimizers.Adam(0.001), loss='mse', metrics=['mae'])

hist = m.fit(
    X_tr_n, y_tr, epochs=100, batch_size=64, validation_split=0.1,
    callbacks=[keras.callbacks.EarlyStopping('val_loss', patience=10, restore_best_weights=True)],
    verbose=0
)

mse, mae = m.evaluate(X_te_n, y_te, verbose=0)
print(f"Test MAE: {mae:.4f} (100k $) = {mae*100:.0f}k $ di errore medio")
print(f"Test RMSE: {mse**0.5:.4f}")
```
</details>

---
### Esercizio 5 (sfida) — API Funzionale di Keras

**Consegna:** L'API Funzionale permette di creare architetture più complesse. Costruisci un modello con due rami che si fondono:

```
Ingresso (10 feature)
       ↓                ↓
Dense(16, relu)    Dense(8, relu)
       ↓                ↓
Dense(8, relu)     Dense(4, relu)
       └──── Concatenare ────┘
              ↓
         Dense(1, sigmoid)
```

1. Usa `keras.Input`, `layers.Dense`, `layers.Concatenate`, `keras.Model`
2. Provalo su make_classification(n_features=10)
3. Confronta con un modello Sequential di architettura simile

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_classification(1000, n_features=10, n_informative=6, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, 0.2, random_state=42)
sc = StandardScaler()
X_tr_n = sc.fit_transform(X_tr)
X_te_n = sc.transform(X_te)

# API Funzionale
tf.random.set_seed(42)
entrada = keras.Input(shape=(10,))

# Ramo 1
rama1 = layers.Dense(16, activation='relu')(entrada)
rama1 = layers.Dense(8,  activation='relu')(rama1)

# Ramo 2
rama2 = layers.Dense(8,  activation='relu')(entrada)
rama2 = layers.Dense(4,  activation='relu')(rama2)

# Fondere
fusionado = layers.Concatenate()([rama1, rama2])
salida    = layers.Dense(1, activation='sigmoid')(fusionado)

modelo_funcional = keras.Model(inputs=entrada, outputs=salida, name='modelo_dos_ramas')
modelo_funcional.summary()

modelo_funcional.compile('adam', 'binary_crossentropy', ['accuracy'])
hist = modelo_funcional.fit(X_tr_n, y_tr, epochs=50, batch_size=32,
                            validation_split=0.2, verbose=0)
_, acc = modelo_funcional.evaluate(X_te_n, y_te, verbose=0)
print(f"Modello a due rami → Test accuracy: {acc:.4f}")

# Confrontare con Sequential
tf.random.set_seed(42)
m_seq = keras.Sequential([
    layers.Input(shape=(10,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8,  activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])
m_seq.compile('adam', 'binary_crossentropy', ['accuracy'])
m_seq.fit(X_tr_n, y_tr, epochs=50, batch_size=32, validation_split=0.2, verbose=0)
_, acc_seq = m_seq.evaluate(X_te_n, y_te, verbose=0)
print(f"Modello Sequential → Test accuracy: {acc_seq:.4f}")
```
</details>

---
## Riepilogo

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# 1. Costruire
modelo = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(n_clases, activation='softmax'),  # o 1+sigmoid per il binario
])

# 2. Compilare
modelo.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # multiclasse con interi
    # loss='categorical_crossentropy'        # multiclasse con one-hot
    # loss='binary_crossentropy'             # binario
    # loss='mse'                             # regressione
    metrics=['accuracy']
)

# 3. Addestrare
hist = modelo.fit(X_tr, y_tr, epochs=50, batch_size=32, validation_split=0.2)

# 4. Valutare e predire
modelo.evaluate(X_te, y_te)
modelo.predict(X_nuevo)

# 5. Salvare
modelo.save('mi_modelo.keras')
modelo = keras.models.load_model('mi_modelo.keras')

# Callback utili
keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
keras.callbacks.ModelCheckpoint('mejor.keras', save_best_only=True)
```

## Cosa viene nella prossima lezione?
Nella **Lezione 5** costruiremo **reti dense** più complete per la classificazione: MNIST in dettaglio, visualizzazione di ciò che impara ogni neurone e confronto tra diverse configurazioni.